# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, explore, and process the Mass Spectrometry EV dataset using the `mlcroissant` library. It follows a FAIR data approach for reproducible analysis on record sets and columns referenced by their `@id` identifiers.

### Dataset Source
*Croissant schema*: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset package and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("\nDataset Title:\n", metadata.name)
print("\nDescription:\n", metadata.description)

# Show main metadata fields
print("\nMetadata fields available:")
for attr in dir(metadata):
    if not attr.startswith("_") and not callable(getattr(metadata, attr)):
        print(f"- {attr}")

## 2. Data Overview
List all available `RecordSet` objects, their `@id`, and the `Field`/`Column` names and `@id` per set.

In [ ]:
# Loop through available record sets in the dataset
print("Available record sets:")
record_sets = []
for record_set in dataset.record_sets():
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_sets.append(record_set.id)
    # List columns/fields
    if hasattr(record_set, 'fields'):
        print("  Fields/Columns:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id})")
    print()
# Preview records from the first record set
if record_sets:
    print(f"Sample rows from {record_sets[0]}:")
    for i, rec in enumerate(dataset.records(record_set=record_sets[0])):
        pprint.pprint(rec)
        if i >= 2: break

## 3. Data Extraction
Extract tabular data from each record set into a DataFrame using record set and field `@id` references.

In [ ]:
# List all record set @ids discovered
print("RecordSet @ids:", record_sets)

# Optionally examine all records sets, but we'll focus on the first by default
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from {record_set_id}...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        dataframes[record_set_id] = df
        # Show head
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Explore numeric fields, filter, normalize, and group the data. All field/column references use their `@id`.

Please change the field ids in the code below after inspecting the available DataFrame columns from step 3.

In [ ]:
# Example: Choose record set and field @ids found above
example_record_set = record_sets[0]
df = dataframes[example_record_set]

# Print all column names
print(f"DataFrame columns: {df.columns.tolist()}")
# Set these variables to a numeric field and group field (by inspecting previous output)
numeric_field = None  # e.g. 'cr:coveragePercent' or another numeric column's @id
group_field = None    # e.g. 'cr:someCategoryField' or another @id

# Try auto-selecting the first float/int column
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field = c
        print(f"Using numeric field: {numeric_field}")
        break

if numeric_field is None:
    print("No numeric field to analyze. Please set 'numeric_field' manually to a column @id.")
else:
    threshold = df[numeric_field].mean()  # Use mean as default threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (N={len(filtered_df)}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a likely categorical field
    for c in df.columns:
        if c != numeric_field and df[c].nunique() < 10:
            group_field = c
            print(f"Using group field: {group_field}")
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Adjust field `@id`s as appropriate.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field and (numeric_field in df.columns):
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30, edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field was detected
    if group_field and group_field in df.columns:
        df.boxplot(column=numeric_field, by=group_field, grid=False, figsize=(8, 4))
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the Mass Spectrometry EV dataset using `mlcroissant`. We demonstrated data access with record set and field `@id`s, performed basic EDA, and visualized numeric fields. You can further adapt this template to your analytical needs and explore other aspects of the dataset or additional record sets.